In [1]:
import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# os.environ["JAX_ENABLE_X64"] = "true"

from pathlib import Path
import numpy as onp
from natsort import natsorted
from typing import Tuple, Callable
import numpy.typing as npt
from scipy.optimize import minimize
import jax
import jax.scipy.optimize
import jax.numpy as jnp
from ase.atoms import Atoms
from ase.visualize import view

from clinamen2.utils.structure_setup import place_atoms_packmol

# Definitions

In [2]:
from jaxborlist.neighbor_list import compute_pairwise_deltas

# see https://gitlab.tuwien.ac.at/e165-03-1_theoretische_materialchemie/scripts-et-al/-/wikis/Sqrt-without-trivial-NaN-derivative-for-jax


@jax.custom_jvp
def _sqrt(x):
    return jnp.sqrt(x)


@_sqrt.defjvp
def _sqrt_jvp(primals, tangents):
    (x,) = primals
    (xdot,) = tangents
    primal_out = _sqrt(x)
    tangent_out = jnp.where(x == 0.0, 0.0, 0.5 / primal_out) * xdot
    return (primal_out, tangent_out)


def wrap_positions(positions, cell, pbcs):
    scaled = positions @ jnp.linalg.pinv(cell)
    wrapped_scaled = jnp.where(pbcs, scaled - scaled // 1.0, scaled)
    return wrapped_scaled @ cell


def shortrange_quadratic_potential(r, r_cut=1.0, prefactor=1.0):
    """Evaluate a short-range quadratic potential for scalar distance arguments

    Args:
        r: Scalar distance value
        r_cut: Cutoff radius of the potential
        prefactor: Prefactor of the potential

    Returns:
        Potential evaluated at distance r
    """
    return jnp.where(
        jnp.abs(r) < r_cut, prefactor * (jnp.abs(r) - r_cut) ** 2, 0.0
    )


def make_overlap_penalty_fn_pairs(
    box_lengths, pbcs, min_dist=1.0, prefactor=1.0
) -> Callable:
    """Create a function that penalizes particles closer together than min_dist

    Args:
        box_lengths: Sequence of box lengths, one value per dimension
        pbcs:
        min_dist: Threshold distance below which particle pair incur an energy
            penalty
        prefactor: Prefactor of the penalty potential

    Returns:
        A function that takes a set of vector-valued particle positions and
        computes the total energy penalty over all particle pairs
    """
    box_lengths = onp.asarray(box_lengths)
    pbcs = onp.asarray(pbcs)

    def energy_fn(positions):
        R_ij = compute_pairwise_deltas(positions, positions, box_lengths, pbcs)
        indices_triu = onp.triu_indices(positions.shape[0], k=1)
        r_ij_triu_2 = (R_ij[indices_triu] * R_ij[indices_triu]).sum(axis=1)
        r_ij_triu = _sqrt(r_ij_triu_2)
        return (
            shortrange_quadratic_potential(
                r_ij_triu, r_cut=min_dist, prefactor=prefactor
            )
        ).sum()

    return energy_fn


def make_minimization_target(energy_fn: Callable, n_dim: int) -> Callable:
    """Make a loss function suitable for scipy optimizers from an energy function

    Args:
        energy_fn: A function that takes in particle positions and returns the
            total system energy.

    Returns:
        A function of two arguments (each argument is one flat array of
        positions) that returns the total energy of the system. The point of
        the two different sets of positions is that the first will be considered
        as degrees of freedom by optimizers, and the second represents any fixed
        particles that might be present.
    """

    def target_fn(positions_dof_flat, positions_fixed_flat):
        positions_dof = positions_dof_flat.reshape((-1, n_dim))
        positions_fixed = positions_fixed_flat.reshape((-1, n_dim))
        positions_combined = jnp.concatenate([positions_dof, positions_fixed])
        return energy_fn(positions_combined)

    return target_fn

In [3]:
def make_optimize_positions(cell, min_dist):
    n_dim = len(cell)

    energy_fn = make_overlap_penalty_fn_pairs(
        box_lengths=onp.diag(cell),
        pbcs=onp.array([True] * n_dim),
        min_dist=min_dist,
    )
    target_fn = make_minimization_target(energy_fn=energy_fn, n_dim=n_dim)

    @jax.jit
    def optimize_positions(positions_dof, positions_fixed):
        result = jax.scipy.optimize.minimize(
            fun=target_fn,
            x0=positions_dof.ravel(),
            args=(positions_fixed.ravel(),),
            method="BFGS",
        )
        return result.x.reshape((-1, n_dim)), result

    return optimize_positions

In [4]:
def postprocess_packmol_structure_for_pbcs(
    positions: npt.ArrayLike,
    cell: npt.ArrayLike,
    min_dist: float,
    optimize_positions_fn: Callable,
) -> Tuple[onp.ndarray, jax.scipy.optimize.OptimizeResults]:
    """Optimize particles near boundaries to avoid overlaps due to periodicity

    This is done by optimizing the positions of the atoms in an "outer shell"
    near the cell boundaries, while keeping atoms in an "inner shell" fixed, and
    ignoring all atoms in the center of the cell (to reduce computational cost).

    Args:
        positions: Array of positions of all particles in the system.
        cell: Unit cell
        min_dist: Minimum distance that particle pairs should have
        optimize_positions_fn: A function that returns optimized outer-shell
            positions, given original outer-shell positions, and inner-shell
            positions as arguments.

    Returns:
        2-element tuple containing:
            - Array of particle positions optimized to remove/reduce overlaps
            - `OptimizeResults` object containing more information about the
                optimization
    """
    approx_thickness_outer_shell = 1.25 * min_dist
    approx_thickness_inner_shell = 1.5 * min_dist

    volume = onp.linalg.det(cell)
    dnsty = positions.shape[0] / volume
    sidelength = cell[0, 0]
    n_dim = positions.shape[1]

    wall_distances = onp.concatenate(
        [onp.abs(positions), onp.abs(onp.diag(cell) - positions)], axis=1
    )
    wall_distances = onp.min(wall_distances, axis=1)
    inds_sorted_by_wall_distance = onp.argsort(wall_distances)

    sidelength_center = sidelength - 2 * (
        approx_thickness_outer_shell + approx_thickness_inner_shell
    )
    sidelength_center_plus_inner = (
        sidelength - 2 * approx_thickness_outer_shell
    )
    volume_center = sidelength_center**3
    volume_center_plus_inner = sidelength_center_plus_inner**3
    n_particles_outer = int(dnsty * (volume - volume_center_plus_inner)) + 1
    n_particles_outer_plus_inner = int(dnsty * (volume - volume_center)) + 1
    inds_outer = inds_sorted_by_wall_distance[:n_particles_outer]
    inds_inner = inds_sorted_by_wall_distance[
        n_particles_outer:n_particles_outer_plus_inner
    ]

    pos_opt_outer_shell, result = optimize_positions_fn(
        positions[inds_outer], positions[inds_inner]
    )

    pos_opt = positions.copy()
    pos_opt[inds_outer] = pos_opt_outer_shell
    pos_opt = wrap_positions(
        pos_opt, cell=cell, pbcs=onp.array([True] * n_dim)
    )

    return onp.asarray(pos_opt), result

In [9]:
DENSITY = 1.0
N_DIM = 3
PACKMOL_TOLERANCE = 2 / 3

OUTDIR_STRUCTURES = Path("structures/")

# Generate structures

In [8]:
OUTDIR_STRUCTURES.mkdir(parents=True)

for n_particles in onp.concatenate(
    [onp.arange(500, 5001, 500), onp.arange(5000, 16000, 1000)]
):
    sidelength = (n_particles / DENSITY) ** (1 / N_DIM)
    box_lengths = jnp.full(N_DIM, sidelength)
    cell = onp.diag(box_lengths)

    optimize_positions = make_optimize_positions(
        cell=cell, min_dist=PACKMOL_TOLERANCE
    )

    positions = []
    charges = []

    for seed in range(11):
        pos = place_atoms_packmol(
            n_atoms=n_particles,
            side_length=sidelength,
            tolerance=PACKMOL_TOLERANCE,
            exec_string="/home/florian/Downloads/packmol-20.14.2/packmol",
            random_seed=seed,
        )
        pos, _ = postprocess_packmol_structure_for_pbcs(
            positions=pos,
            cell=cell,
            min_dist=PACKMOL_TOLERANCE,
            optimize_positions_fn=optimize_positions,
        )
        positions.append(pos)

        rng = onp.random.default_rng(seed)
        chg = rng.uniform(low=-1.0, high=1.0, size=n_particles)
        chg -= chg.sum() / n_particles
        charges.append(chg)

    outdict = {
        "positions": onp.asarray(positions),
        "charges": onp.asarray(charges),
        "cells": onp.stack([cell] * len(positions)),
    }

    onp.savez(OUTDIR_STRUCTURES / f"structures_{n_particles}", **outdict)

tolerance 0.6666666666666666
filetype xyz 
output box_15000.xyz
seed 0

structure trivial.xyz
  number 15000
  inside box 0. 0. 0. 24.662120743304698 24.662120743304698 24.662120743304698
end structure


################################################################################

 PACKMOL - Packing optimization for the automated generation of
 starting configurations for molecular dynamics simulations.
 
                                                              Version 20.14.2 

################################################################################

  Packmol must be run with: packmol < inputfile.inp 

  Userguide at: http://m3g.iqm.unicamp.br/packmol 

  Reading input file... (Control-C aborts)
  Types of coordinate files specified: xyz
  Seed for random number generator:            0
  Output file: box_15000.xyz
  Reading coordinate file: trivial.xyz
  Number of independent structures:            1
  The structures are: 
  Structure            1 :sphere(           

# Make checks on the generated structures

In [10]:
def find_min_pair_distance_ase(atoms: Atoms, mic=True):
    all_distances = atoms.get_all_distances(mic=mic)
    all_distances[onp.diag_indices_from(all_distances)] = onp.inf
    return all_distances.min()

In [11]:
def batched_find_min_pair_distance(positions, box_lengths, pbcs, batch_size):
    n_particles = positions.shape[0]
    n_full_batches = n_particles // batch_size

    nruter = jnp.inf

    for i in range(n_full_batches + 1):
        start = i * batch_size
        end = min((i + 1) * batch_size, n_particles)
        if start >= end:
            continue
        deltas = compute_pairwise_deltas(
            positions[start:end], positions, box_lengths=box_lengths, pbcs=pbcs
        )
        squared_distances = (deltas * deltas).sum(axis=2)
        self_mask = onp.arange(start, end)[:, jnp.newaxis] == onp.arange(
            n_particles
        )
        squared_distances = jnp.where(self_mask, jnp.inf, squared_distances)
        nruter = min(nruter, squared_distances.min())

    return jnp.sqrt(nruter)

In [16]:
def batched_find_max_neighbors_jit(
    positions, r_cut, box_lengths, pbcs, batch_size
):
    n_particles = positions.shape[0]
    n_full_batches = n_particles // batch_size

    @jax.jit
    def get_max_neighbors_one_batch(pos, batch_inds):
        deltas = compute_pairwise_deltas(
            pos[batch_inds], pos, box_lengths=box_lengths, pbcs=pbcs
        )
        squared_distances = (deltas * deltas).sum(axis=2)
        self_mask = batch_inds[:, jnp.newaxis] == onp.arange(n_particles)
        squared_distances = jnp.where(self_mask, jnp.inf, squared_distances)
        max_neighbors = (squared_distances < r_cut * r_cut).sum(axis=1).max()
        return max_neighbors

    max_neighbors_over_batches = []
    for i in range(n_full_batches + 1):
        start = i * batch_size
        end = min((i + 1) * batch_size, n_particles)
        if start >= end:
            continue
        max_neighbors_this_batch = get_max_neighbors_one_batch(
            pos=positions, batch_inds=jnp.arange(start, end)
        )
        max_neighbors_over_batches.append(max_neighbors_this_batch)

    return jnp.max(jnp.array(max_neighbors_over_batches))


def batched_find_max_neighbors_and_min_pair_dists(
    positions, r_cut, box_lengths, pbcs, batch_size
):
    n_particles = positions.shape[0]
    n_full_batches = n_particles // batch_size

    @jax.jit
    def process_one_batch(pos, batch_inds):
        deltas = compute_pairwise_deltas(
            pos[batch_inds], pos, box_lengths=box_lengths, pbcs=pbcs
        )
        squared_distances = (deltas * deltas).sum(axis=2)
        self_mask = batch_inds[:, jnp.newaxis] == onp.arange(n_particles)
        squared_distances = jnp.where(self_mask, jnp.inf, squared_distances)
        max_neighbors = (squared_distances < r_cut * r_cut).sum(axis=1).max()
        min_pair_distance = jnp.sqrt(squared_distances.min())
        return max_neighbors, min_pair_distance

    max_neighbors_over_batches = []
    min_pair_distances_over_batches = []
    for i in range(n_full_batches + 1):
        start = i * batch_size
        end = min((i + 1) * batch_size, n_particles)
        if start >= end:
            continue
        max_neighbors_this_batch, min_pair_dist_this_batch = process_one_batch(
            pos=positions, batch_inds=jnp.arange(start, end)
        )
        max_neighbors_over_batches.append(max_neighbors_this_batch)
        min_pair_distances_over_batches.append(min_pair_dist_this_batch)

    max_neighbors = jnp.max(jnp.array(max_neighbors_over_batches))
    min_pair_distance = jnp.min(jnp.array(min_pair_distances_over_batches))

    return max_neighbors, min_pair_distance

In [ ]:
def batched_find_max_neighbors_jit(
    positions, r_cut, box_lengths, pbcs, batch_size
):
    n_particles = positions.shape[0]
    n_full_batches = n_particles // batch_size

    @jax.jit
    def get_max_neighbors_one_batch(pos, batch_inds):
        deltas = compute_pairwise_deltas(
            pos[batch_inds], pos, box_lengths=box_lengths, pbcs=pbcs
        )
        squared_distances = (deltas * deltas).sum(axis=2)
        self_mask = batch_inds[:, jnp.newaxis] == onp.arange(n_particles)
        squared_distances = jnp.where(self_mask, jnp.inf, squared_distances)
        max_neighbors = (squared_distances < r_cut * r_cut).sum(axis=1).max()
        return max_neighbors

    max_neighbors_over_batches = []
    for i in range(n_full_batches + 1):
        start = i * batch_size
        end = min((i + 1) * batch_size, n_particles)
        if start >= end:
            continue
        max_neighbors_this_batch = get_max_neighbors_one_batch(
            pos=positions, batch_inds=jnp.arange(start, end)
        )
        max_neighbors_over_batches.append(max_neighbors_this_batch)

    return jnp.max(jnp.array(max_neighbors_over_batches))

In [17]:
pbc = onp.array([True] * N_DIM)

for npzfile in natsorted(OUTDIR_STRUCTURES.glob("structures_*.npz")):
    print(npzfile)

    loaded = onp.load(npzfile)

    # All structures of the same particle number were created in the same cell
    cell = loaded["cells"][0]

    energy_fn = make_overlap_penalty_fn_pairs(
        box_lengths=onp.diag(cell),
        pbcs=pbc,
        min_dist=PACKMOL_TOLERANCE,
    )
    energy_fn = jax.jit(energy_fn)

    for idx_structure in range(len(loaded["positions"])):
        pos = loaded["positions"][idx_structure]
        chg = loaded["charges"][idx_structure]

        e = energy_fn(pos)
        (
            n_max_nghbrs,
            min_dist,
        ) = batched_find_max_neighbors_and_min_pair_dists(
            positions=pos,
            r_cut=PACKMOL_TOLERANCE,
            box_lengths=onp.diag(cell),
            pbcs=pbc,
            batch_size=1000,
        )
        if min_dist < PACKMOL_TOLERANCE - 0.01:
            print(f"- min_dist = {min_dist:.3f}")

    print()

structures/structures_500.npz

structures/structures_1000.npz

structures/structures_1500.npz

structures/structures_2000.npz

structures/structures_2500.npz

structures/structures_3000.npz

structures/structures_3500.npz

structures/structures_4000.npz

structures/structures_4500.npz

structures/structures_5000.npz
- min_dist = 0.625

structures/structures_6000.npz

structures/structures_7000.npz

structures/structures_8000.npz

structures/structures_9000.npz

structures/structures_10000.npz

structures/structures_11000.npz

structures/structures_12000.npz

structures/structures_13000.npz

structures/structures_14000.npz

structures/structures_15000.npz



## Visual inspection

In [18]:
from ase.atoms import Atoms
from ase.visualize import view

In [19]:
loaded = onp.load(OUTDIR_STRUCTURES / "structures_4000.npz")

In [20]:
idx_structure = 3
atoms = Atoms(
    cell=loaded["cells"][idx_structure],
    positions=loaded["positions"][idx_structure],
)

In [21]:
view(atoms)

In [22]:
find_min_pair_distance_ase(atoms)

0.6668872357851219

In [24]:
batched_find_max_neighbors_and_min_pair_dists(
    atoms.positions,
    r_cut=PACKMOL_TOLERANCE,
    box_lengths=onp.diag(atoms.get_cell()[...]),
    pbcs=pbc,
    batch_size=1000,
)

(Array(0, dtype=int32), Array(0.6668873, dtype=float32))